# 04 — Training, loss balancing, and optimization

Study the optimization layer of the PINN: composite losses, weight sensitivity, reproducibility, convergence, and an Adam → L-BFGS refinement stage.

In [ ]:
import torch
import matplotlib.pyplot as plt
from pinn import MLP, PINNConfig, PINNTrainer, sample_heat_equation
torch.set_default_dtype(torch.float32)
alpha=0.1
points=sample_heat_equation(2500,500,500,seed=123)


## 1. Baseline training

In [ ]:
trainer=PINNTrainer(MLP(hidden_dim=48,hidden_layers=3),PINNConfig(alpha=alpha,initial_weight=10,boundary_weight=10,learning_rate=1e-3,epochs=1500,log_every=300,seed=123))
history=trainer.train(points)
print('final total:',history.total[-1])


In [ ]:
plt.figure(figsize=(8,4))
for k,v in [('total',history.total),('physics',history.physics),('initial',history.initial),('boundary',history.boundary)]: plt.semilogy(v,label=k)
plt.xlabel('epoch'); plt.ylabel('loss'); plt.legend(); plt.show()


## 2. Weight sensitivity

The same point set is used so that the main changing variable is the relative weighting of the physical and constraint terms.

In [ ]:
experiments={'balanced':(1,10,10),'strong_constraints':(1,50,50),'physics_heavy':(5,5,5)}
results={}
for name,(wf,w0,wb) in experiments.items():
    tr=PINNTrainer(MLP(hidden_dim=40,hidden_layers=3),PINNConfig(alpha=alpha,physics_weight=wf,initial_weight=w0,boundary_weight=wb,epochs=700,seed=11))
    results[name]=tr.train(points)
    h=results[name]
    print(name,h.total[-1],h.physics[-1],h.initial[-1],h.boundary[-1])


In [ ]:
plt.figure(figsize=(8,4))
for name,h in results.items(): plt.semilogy(h.total,label=name)
plt.xlabel('epoch'); plt.ylabel('total loss'); plt.legend(); plt.show()


## 3. Reproducibility check

In [ ]:
a=PINNTrainer(MLP(hidden_dim=24,hidden_layers=2),PINNConfig(alpha=alpha,epochs=8,seed=77)); ha=a.train(points)
b=PINNTrainer(MLP(hidden_dim=24,hidden_layers=2),PINNConfig(alpha=alpha,epochs=8,seed=77)); hb=b.train(points)
print('same history:',ha.total==hb.total)


## 4. Adam followed by L-BFGS

L-BFGS is a full-batch quasi-Newton stage. The trainer implements it as an optional refinement after Adam.

In [ ]:
refined=PINNTrainer(MLP(hidden_dim=48,hidden_layers=3),PINNConfig(alpha=alpha,initial_weight=20,boundary_weight=20,epochs=700,learning_rate=1e-3,use_lbfgs=True,lbfgs_steps=40,seed=5))
rh=refined.train(points)
print('Adam-stage final:',rh.total[-1])


## 5. Convergence must be evaluated with physics and reference metrics

A decreasing training objective alone does not establish physical correctness. Evaluate independent residuals and, where available, analytical error.

In [ ]:
from pinn import heat_exact_solution, heat_residual
x=torch.linspace(-1,1,120); t=torch.linspace(0,1,80)
xx,tt=torch.meshgrid(x,t,indexing='ij'); grid=torch.stack([xx.reshape(-1),tt.reshape(-1)],1)
with torch.no_grad(): pred=refined.predict(grid)
exact=heat_exact_solution(grid[:,0:1],grid[:,1:2],alpha)
err=pred-exact
print('RMSE:',float(torch.sqrt(torch.mean(err.square()))))
print('relative L2:',float(torch.linalg.vector_norm(err)/torch.linalg.vector_norm(exact)))
r=heat_residual(refined.model,grid.clone().requires_grad_(True),alpha)
print('PDE residual RMSE:',float(torch.sqrt(torch.mean(r.square()))))
